# Evaluating the Warehouse Agent with Strands Evals & AgentCore Evaluations

In Labs 7 and 8 you deployed a warehouse operations agent to Amazon Bedrock AgentCore Runtime and gave it memory. But an answer that *looks* right is not proof the agent is good. Before you trust it with real procurement decisions, you need to measure its quality in a repeatable way.

This lab builds an evaluation suite and grades the agent in **two stages**. Stage 1 runs the agent locally, inside this notebook, with [Strands Evals](https://pypi.org/project/strands-agents-evals/), so you get fast, cheap feedback while you iterate. Stage 2 grades the agent already running on AgentCore Runtime with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/), which reads the production traces the deployed agent emits. The grader design follows Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents).

> **This notebook runs on its own.** The setup section below sets up everything it needs again (the model, the local agent, and the deployment details), so you do not need Lab 8 loaded in the same kernel. Stage 1 needs only your SAP GenAI Hub credentials. Stage 2 also needs the runtime you deployed in Lab 7 and gave memory in Lab 8.

Note: If you have not set up your ai-core credentials yet, please follow notebook 00-load-sap-ai-core-credentials.


## Prerequisites

1. **Completed Lab 7**, which deploys a warehouse agent to AgentCore Runtime and writes `lab7_deployment.json`. Required for Stage 2.
2. **Completed Lab 8** (recommended), so the deployed runtime has memory. Stage 2 grades whatever is currently deployed.
3. SAP AI Core credentials in `~/.aicore/config.json` (run Lab 00 if you have not).
4. An SAP S/4HANA Public Cloud API key, either in `.env` or entered when prompted.
5. AWS credentials with AgentCore Evaluation permissions.

Stage 1 needs only items 1, 3, and 4. Stage 2 also needs the deployed runtime (item 2) and AWS permissions (item 5).


## 1. Setup: imports, configuration, and the local agent

This lab is self-contained. The next few cells import dependencies, check your credentials, initialize the `SAPGenAIHubModel`, load the deployed-agent details from `lab7_deployment.json`, and build the local warehouse agent used in Stage 1. None of this depends on Lab 8 having run in the same kernel.


In [ ]:
from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel
from util.odata_tool import odata_caller
from strands import Agent, tool

import os
import json
import time
import uuid
from datetime import datetime, timedelta
from collections import defaultdict

import boto3
from dotenv import load_dotenv
import getpass

# Load environment variables from .env file
load_dotenv()

# Prompt for the SAP API key if it isn't already set
if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    os.environ["SAP_S4HANA_PUBLIC_CLOUD_KEY"] = getpass.getpass("SAP_S4HANA_PUBLIC_CLOUD_KEY:\n")


In [ ]:
# Validate required configuration before proceeding
import os

_errors = []

if not os.path.exists(os.path.expanduser("~/.aicore/config.json")):
    _errors.append("Missing ~/.aicore/config.json — run notebook 00 first")

if not os.environ.get("SAP_S4HANA_PUBLIC_CLOUD_KEY"):
    _errors.append("SAP_S4HANA_PUBLIC_CLOUD_KEY not set — check your .env file")

try:
    _sts = boto3.client("sts").get_caller_identity()
    print(f"AWS Identity: {_sts['Arn']}")
except Exception as e:
    _errors.append(f"AWS credentials not configured: {e}")

if _errors:
    for err in _errors:
        print(f"ERROR: {err}")
    raise SystemExit("Fix the errors above before continuing.")
else:
    print("All prerequisites validated.")

In [ ]:
# TODO: Choose your model — options: "anthropic--claude-4.5-sonnet", "amazon--nova-lite", "amazon--nova-pro"
model = SAPGenAIHubModel(
    model_id="anthropic--claude-4.5-sonnet",
    max_tokens=4096,
)

# Load the deployed-agent details written by Lab 7 (lab7_deployment.json). Stage 2 evaluates
# THIS deployed runtime. Stage 1 (local) only needs `model`, but we load the deployment up
# front so the whole lab is configured in one place.
DEPLOYMENT_FILE = "lab7_deployment.json"
if not os.path.exists(DEPLOYMENT_FILE):
    raise SystemExit(
        f"{DEPLOYMENT_FILE} not found — run Lab 7 "
        "(07-deploy-warehouse-agent-to-agentcore.ipynb) first (and Lab 8 to add memory)."
    )

with open(DEPLOYMENT_FILE, "r") as f:
    deployment = json.load(f)

AGENT_NAME = deployment["agent_name"]
AGENT_ID = deployment["agent_id"]
AGENT_ARN = deployment["agent_arn"]
REGION = deployment.get("region") or "us-east-1"

print(f"Region: {REGION}")
print(f"Agent Name: {AGENT_NAME}")
print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"Model: {model.get_config()['model_id']} via SAP GenAI Hub")


In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from botocore.exceptions import ClientError

MEMORY_NAME = "WarehouseAgentMemory"  # must match Lab 8
ACTOR_ID = "warehouse_manager_001"    # must match Lab 8

memory_client = MemoryClient(region_name=REGION)

_strategies = [
    {StrategyType.SEMANTIC.value: {
        "name": "WarehouseFacts",
        "description": "Operational facts about warehouse inventory, orders, and logistics",
        "namespaces": ["warehouse/{actorId}/facts/"],
    }},
    {StrategyType.USER_PREFERENCE.value: {
        "name": "UserPreferences",
        "description": "User preferences: reporting format, products of interest, alert thresholds",
        "namespaces": ["warehouse/{actorId}/preferences/"],
    }},
]


def ensure_memory():
    """Create the memory resource, or find and reuse it if it already exists."""
    try:
        mem = memory_client.create_memory_and_wait(
            name=MEMORY_NAME,
            description="Memory for warehouse operations agent",
            strategies=_strategies,
            event_expiry_days=30,
        )
        print(f"Created memory resource: {mem['id']}")
        return mem["id"]
    except ClientError as e:
        if "already exists" in str(e):
            mid = next(
                (m["id"] for m in memory_client.list_memories() if m["id"].startswith(MEMORY_NAME)),
                None,
            )
            print(f"Reusing existing memory resource: {mid}")
            return mid
        raise


memory_id = ensure_memory()
if not memory_id:
    raise SystemExit("Could not create or find the memory resource.")

### The local agent

Stage 1 grades a local copy of the *same* warehouse agent you built in Lab 6 and deployed in Labs 7 and 8. It is a two-agent design: a **selector sub-agent** (`SelectorAPIAgentAsATool`) reads the OpenAPI specs under `assets/knowledgebase` and picks which SAP API fits the question, then the **warehouse agent** calls that selector and the `odata_caller` tool to fetch inventory data and answer.

Running it here, in process, lets us read its tool calls straight off the result. Two things to know:

- **It runs without memory.** Stage 1 is about the agent's query logic, so we grade it memory-free. The memory-enabled deployed runtime is graded in Stage 2.
- **The baseline prompt is deliberately naive.** It tells the agent to discover the schema at runtime (call the selector, then read `$metadata`). That is flexible but wasteful, and it is exactly the inefficiency Stage 1's trajectory judge will catch. Later in Stage 1 we swap in an improved prompt and re-measure.

The agent and both prompts live in `util/warehouse_agent.py`, extracted from Lab 6 so this notebook stays focused on evaluation.


In [ ]:
# The Lab 6 warehouse agent (selector sub-agent + odata_caller), its system prompts, and the
# OData schema hints live in util/warehouse_agent.py so this notebook stays focused on
# evaluation. We import two prompts: the baseline (Lab 6/7/8 — rediscovers the schema at
# runtime via the selector + $metadata) and an improved one (which knows the entity and fields).
from util.warehouse_agent import (
    WAREHOUSE_SYSTEM_PROMPT,
    WAREHOUSE_SYSTEM_PROMPT_IMPROVED,
    WAREHOUSE_CAPACITY,
    build_warehouse_agent,
)


def create_warehouse_agent(with_memory=False, system_prompt=WAREHOUSE_SYSTEM_PROMPT):
    """Local (memory-free) warehouse agent for Stage 1. Pass the improved prompt to grade the fix."""
    return build_warehouse_agent(model, system_prompt=system_prompt)


# Smoke-test that the local agent constructs (builds the selector sub-agent + warehouse agent).
_ = create_warehouse_agent()
print("Local warehouse agent ready for Stage 1 evaluation.")


## How we evaluate: two stages

The two stages answer two different questions:

1. **Stage 1, local, with [Strands Evals](https://pypi.org/project/strands-agents-evals/) — "is the logic good?"** Run the agent in this notebook and grade it in process. Fast, cheap, and no deployment needed. This is the pre-deploy gate you iterate and *fix* against before you ship.
2. **Stage 2, deployed, with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/) — "did the fix ship, and how does the real thing behave under production infrastructure?"** We ship Stage 1's fix to the live AgentCore Runtime, then grade it from the production OpenTelemetry (OTEL) traces it writes to CloudWatch. That means a **parity** check (did quality hold?) plus **prod-only signals** a local run physically can't produce — latency, cold-start, and cost. This is also what you wire into ongoing production monitoring.

Grade locally as you build, then ship the fix and verify it in production.

```
  STAGE 1 — Local (this notebook)                STAGE 2 — Deployed (AgentCore Runtime)
  ┌───────────────────────────────┐             ┌────────────────────────────────────────┐
  │ create_warehouse_agent()      │             │ redeploy improved prompt (+ memory)       │
  │        │ run scenario         │             │        │ invoke_agent_runtime() ─OTEL─► CW  │
  │        ▼                      │             │        ▼   EvaluationClient.run()          │
  │ Strands Evals                 │             │ AgentCore built-in + custom evaluators    │
  │  • ToolCalled (code)          │             │  • Correctness / Helpfulness (LLM-judge)  │
  │  • OutputEvaluator (LLM)      │             │  • GoalSuccessRate / ToolSelectionAccuracy│
  │  • TrajectoryEvaluator (LLM)  │             │  • WarehouseOperationalQuality (custom)   │
  │  • ToolSelectionAccuracy (LLM)│             │  + prod-only: latency · cold-start · cost │
  └───────────────────────────────┘             └────────────────────────────────────────┘
     is the logic good? (pre-deploy gate)          did the fix ship? how does prod behave?
```


## Understanding grader types

Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) groups graders into three families. Most real evaluations **combine** them.

| Grader family | How it works | In this lab |
|---|---|---|
| **Code-based** | Deterministic checks: string match, counting, static analysis | `ToolCalled` (local) |
| **Model-based (LLM as judge)** | An LLM scores the output against a rubric | `OutputEvaluator`, `TrajectoryEvaluator`, `ToolSelectionAccuracy` (local); `Correctness`, `Helpfulness`, `GoalSuccessRate`, `ToolSelectionAccuracy`, and the custom `WarehouseOperationalQuality` (deployed) |
| **Human** | Expert review and spot checks | Not shown here; used in practice to calibrate the LLM judges |

Rule of thumb: use code-based graders wherever you can (cheapest and most reliable), reach for LLM judges when you need nuance, and use humans to calibrate those judges.

**There is a second question to ask about every grader: *what* is it grading?**

- **Outcome** grades the final answer. *Did the agent get it right?* Examples: `OutputEvaluator` (local), `Correctness` and `WarehouseOperationalQuality` (deployed).
- **Trajectory** grades the path taken. *Did it use the right tools, efficiently?* Examples: `ToolCalled`, `TrajectoryEvaluator`, `ToolSelectionAccuracy`.

You want both. Our `low-stock-items` scenario shows why: the agent reaches the correct answer (outcome passes) but takes a wasteful path of redundant OData calls (trajectory fails). An outcome-only score would call that a clean pass and hide the problem.


## Evaluation scenarios

These scenarios are **shared by both stages**: they are graded locally in Stage 1 and on the deployed runtime in Stage 2. Each scenario carries four things:

- **prompt**: the user query the agent must answer.
- **expected_response**: what a good answer looks like, used by the outcome graders (`OutputEvaluator` locally, `Correctness` and the custom evaluator when deployed).
- **expected_trajectory**: the tool calls we expect, used by the trajectory graders (`ToolSelectionAccuracy`, `TrajectoryEvaluator`).
- **assertions**: the goal conditions, used by `GoalSuccessRate` in Stage 2.

The last scenario, `low-stock-items`, is a real workshop failure. The query triggers many redundant OData calls (repeated `$metadata` reads, trial-and-error `$filter` guessing) yet still returns a correct answer. Outcome graders score it 1.0, so the inefficiency is invisible to them. The `TrajectoryEvaluator` in Stage 1 is what catches the wasteful path and explains it.


In [ ]:
evaluation_scenarios = [
    {
        "name": "inventory-check-single",
        "prompt": "What is the current stock level for WM-AN02 Control Units?",
        "expected_response": (
            "The response should contain the specific stock quantity for WM-AN02 Control Units "
            "retrieved from the SAP warehouse API, with the product correctly identified as "
            "Control Units. The number should come from actual API data, not be invented."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent correctly identified WM-AN02 as Control Units. "
            "Agent reported a specific numeric stock quantity from the API."
        ),
    },
    {
        "name": "inventory-overview",
        "prompt": "Give me a complete overview of all products currently in the warehouse.",
        "expected_response": (
            "The response should list all warehouse products (WM-AN01 Advanced Sensors, "
            "WM-AN02 Control Units, WM-AN03 Power Modules, WM-AN04 Communication Devices) "
            "with their current stock quantities from the API, presented in a structured format."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via OData. "
            "Agent listed multiple products with stock quantities. "
            "Agent presented results in a structured, readable format."
        ),
    },
    {
        "name": "fulfillment-feasibility",
        "prompt": "Can we fulfill an order for 200 units of WM-AN02 Control Units?",
        "expected_response": (
            "The response should check current WM-AN02 stock from the API, compare it against "
            "the requested 200 units, and provide a clear yes/no fulfillment recommendation "
            "with the actual available quantity."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried current WM-AN02 stock via OData. "
            "Agent compared available quantity against the 200-unit request. "
            "Agent provided a clear yes/no fulfillment answer with supporting data."
        ),
    },
    {
        # Regression scenario drawn from a real workshop failure. This query used to trigger
        # many redundant OData calls (repeated $metadata discovery, trial-and-error $filter
        # guessing). Correctness/GoalSuccessRate scored it 1.0 despite the waste, so we lean on
        # the LLM-as-judge trajectory grader (Stage 1) to catch — and explain — the inefficiency.
        "name": "low-stock-items",
        "prompt": "What are the items with low stock?",
        "expected_response": (
            "The response should identify which products are low on stock (or state that none "
            "are below the reorder threshold), based on stock quantities from the SAP warehouse "
            "API. The agent should reach the answer efficiently — ideally a single filtered or "
            "sorted OData query — rather than fetching everything and repeatedly rediscovering the schema."
        ),
        "expected_trajectory": ["odata_caller"],
        "assertions": (
            "Agent queried warehouse stock data via the OData API. "
            "Agent determined which products are low on stock relative to their capacity. "
            "Agent reached the answer without redundant, repeated OData calls."
        ),
    },
]

print(f"Evaluation scenarios defined: {len(evaluation_scenarios)}")
for s in evaluation_scenarios:
    print(f"  - {s['name']}: {s['prompt']}")

## Stage 1 — Local evaluation with Strands Evals

Run the agent right here in the notebook and grade it in process. This is the fast, cheap gate you iterate against before deploying. **[Strands Evals](https://pypi.org/project/strands-agents-evals/)** (already a project dependency) gives us graders from both families:

- **Code-based (deterministic):** `ToolCalled` checks that the agent used its tool at all.
- **LLM as judge:** `OutputEvaluator` scores whether the final answer is correct (task success), `TrajectoryEvaluator` scores the tool-call path against a rubric, and `ToolSelectionAccuracyEvaluator` judges whether each individual call was justified. All three route through the **same `SAPGenAIHubModel`** the agent uses, so no separate Bedrock access is needed.

**Where the trajectory comes from.** Each scenario runs once against the memory-free local agent. We read its tool usage off the result with the SDK's `tools_use_extractor` ([docs](https://strandsagents.com/docs/user-guide/evals-sdk/quickstart/)):

```python
from strands_evals.extractors import tools_use_extractor
calls = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
# -> [{"name": "odata_caller", "input": {...}, "tool_result": "...", "is_error": False}, ...]
```

We keep two views of that run: a flat list of tool names for `ToolCalled`, and a detailed list (each call's OData arguments plus an `is_error` flag) for the trajectory judge, so it can name the *specific* redundant or failed calls instead of just counting them. This is what lets a scenario **pass on task success but score low on trajectory**: the right answer reached the wrong way. That is the `low-stock-items` case, and it is exactly what an outcome-only score misses.


In [ ]:
# Code-based grader with Strands Evals — deterministic, no LLM. This cell also defines the
# reusable `evaluate_agent` helper the "fix and re-test" cell below calls a second time.
from strands_evals.evaluators import ToolCalled
from strands_evals.extractors import tools_use_extractor
from strands_evals.types import EvaluationData

# Name of the tool we count. TODO: change if your agent uses a different tool.
TOOL_NAME = "odata_caller"

# OData argument keys worth surfacing to the trajectory judge — these reveal waste
# (repeated $metadata rediscovery, trial-and-error $filter guessing).
_ODATA_KEYS_OF_INTEREST = ("endpoint", "operation", "$filter", "$orderby", "$select", "$top")

# Captured once so the trajectory judge knows what each tool does (see the LLM-judge cell).
TOOL_DESCRIPTIONS = {}


def _summarize_tool_call(call: dict) -> dict:
    """Reduce one extracted tool-call record to the fields that show *what* it did.

    `tools_use_extractor` gives {name, input, tool_result, is_error} per call. We surface the
    endpoint and OData query params (so the judge sees $filter/$orderby directly) and keep
    is_error, since a failed call is the tell-tale of trial-and-error $filter guessing.
    """
    tool_input = call.get("input") or {}
    odata_params = tool_input.get("odata_params") or {}
    flat = {**tool_input, **odata_params}
    summary = {"name": call.get("name")}
    for key in _ODATA_KEYS_OF_INTEREST:
        if flat.get(key):
            summary[key] = flat[key]
    if call.get("is_error"):
        summary["is_error"] = True
    return summary


def run_local_agent_trajectory(prompt: str, make_agent):
    """Run one scenario against a local agent; return (output, names, detailed, tokens).

    - names    — flat tool-name list, for `ToolCalled` (exact-match membership).
    - detailed — ordered {name, endpoint, $filter, is_error, ...} dicts, for the LLM judge.
    - tokens   — total tokens the agent-under-test consumed (informational efficiency signal).
    Both trajectory views come from the SDK's `tools_use_extractor` (no hand-parsing).
    """
    global TOOL_DESCRIPTIONS
    agent = make_agent()
    result = agent(prompt)
    output_text = str(result.message)

    if not TOOL_DESCRIPTIONS:
        TOOL_DESCRIPTIONS = tools_use_extractor.extract_tools_description(agent)

    calls = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
    names = [c.get("name") for c in calls]
    detailed = [_summarize_tool_call(c) for c in calls]
    tokens = (result.metrics.accumulated_usage or {}).get("totalTokens")
    return output_text, names, detailed, tokens


def evaluate_agent(make_agent, scenarios):
    """Run every scenario against `make_agent()` once and grade with the code-based `ToolCalled`.

    Returns {scenario_name: {names, detailed, output, tokens, name_case, detail_case,
    tool_called_result}} — the per-scenario cache reused by the LLM-judge cell and the
    before/after comparison, so we invoke the agent once per scenario, not once per grader.
    """
    runs = {}
    for scenario in scenarios:
        name = scenario["name"]
        output_text, names, detailed, tokens = run_local_agent_trajectory(scenario["prompt"], make_agent)

        # name_case carries expected_output so the OutputEvaluator (task-success judge) can
        # compare the answer against what a good response should contain.
        name_case = EvaluationData(
            name=name, input=scenario["prompt"], actual_output=output_text,
            expected_output=scenario["expected_response"],
            actual_trajectory=names, expected_trajectory=scenario["expected_trajectory"],
        )
        detail_case = EvaluationData(
            name=name, input=scenario["prompt"], actual_output=output_text,
            actual_trajectory=detailed, expected_trajectory=scenario["expected_trajectory"],
        )

        tool_called = ToolCalled(TOOL_NAME).evaluate(name_case)[0]
        n_calls = names.count(TOOL_NAME)
        runs[name] = {
            "name_case": name_case,
            "detail_case": detail_case,
            "names": names,
            "detailed": detailed,
            "output": output_text,
            "tokens": tokens,
            "tool_called_result": {
                "evaluatorId": "StrandsEvals.ToolCalled",
                "value": tool_called.score,
                "label": "called" if tool_called.test_pass else "not_called",
                "explanation": tool_called.reason,
                "n_tool_calls": n_calls,
                "tool_called": tool_called.test_pass,
            },
        }
        tok = f"{tokens} tokens" if tokens is not None else "tokens n/a"
        print(f"  {name}: tool_called={tool_called.test_pass} ({n_calls} '{TOOL_NAME}' call(s), {tok})")
    return runs


print("Running Strands Evals code-based grader on the baseline agent...\n")
local_runs = evaluate_agent(lambda: create_warehouse_agent(with_memory=False), evaluation_scenarios)
tool_called_results = {n: [r["tool_called_result"]] for n, r in local_runs.items()}
print("\nCode-based grading complete.")

### LLM-as-judge graders (local)

`ToolCalled` tells you the agent reached its tool, not whether it answered well or worked efficiently. For that we add three LLM-as-judge graders, all routed through the same `SAPGenAIHubModel`:

- **`OutputEvaluator`** (the outcome judge, [docs](https://strandsagents.com/docs/user-guide/evals-sdk/evaluators/output_evaluator/)) reads the agent's final answer and the scenario's `expected_response` and scores **task success** on content alone: did the agent return the right inventory data, with the right product codes, without inventing numbers? This is the "was it actually right?" metric.
- **`TrajectoryEvaluator`** scores the tool-call path against a rubric. Fed the detailed trajectory (endpoints, OData arguments, `is_error` flags), it names the specific redundant or failed calls, so its reason is a diagnosis you can act on.
- **`ToolSelectionAccuracyEvaluator`** is a call-level judge ("was this call justified?"). It needs a Strands `Session` built from OTEL spans, so we drive it through the `Experiment` and `TracedHandler` harness, which captures spans from a fresh local run.

Together the outcome judge and the trajectory judge answer two different questions about the same run: **was the answer right, and was the path to it efficient?**

> These graders make **live LLM calls**, so this cell is slower than the deterministic grader and scores can vary slightly between runs. The `ToolSelectionAccuracyEvaluator` path re-runs the agent to capture traces; if local span capture is unavailable it is skipped and the `OutputEvaluator` and `TrajectoryEvaluator` results still stand.


In [ ]:
# LLM-as-judge graders with Strands Evals — routed through the same SAPGenAIHubModel.
from strands_evals.evaluators import (
    OutputEvaluator,
    TrajectoryEvaluator,
    ToolSelectionAccuracyEvaluator,
)

# Rubric for the OUTPUT judge. This is the "did the agent actually succeed?" grader: it reads the
# final answer and the scenario's expected_response and scores task success on content alone,
# ignoring the path taken. Pairs with the trajectory judge below (right answer vs. right process).
OUTPUT_RUBRIC = (
    "You are grading a warehouse inventory agent's final answer for TASK SUCCESS. Compare the "
    "agent's output against the expected response, judging factual content only — ignore wording, "
    "formatting, and style. A successful answer contains the specific inventory data the question "
    "asked for (stock quantities, correct product codes such as WM-AN02, a clear yes/no where a "
    "decision was requested) and does not invent numbers.\n\n"
    "Score 1.0 if the answer fully satisfies the expected response, 0.5 if it is partially correct "
    "or missing key data, and 0.0 if it is wrong, empty, or hallucinated. State briefly in your "
    "reason which required facts are present or missing."
)

# Rubric for the TRAJECTORY judge. Efficiency is what we care about for these scenarios, and —
# crucially — we ask the judge to NAME the specific redundant calls it sees, so its `reason`
# becomes an actionable diagnosis rather than just a score.
TRAJECTORY_RUBRIC = (
    "Score the tool-call trajectory for a warehouse inventory agent. Each entry shows the tool "
    "name and the OData arguments used ($filter, $orderby, endpoint, etc.); entries with "
    "`is_error: true` are calls that FAILED. A good trajectory answers the user's question with "
    "as few tool calls as possible — ideally one filtered or sorted OData query.\n\n"
    "Penalize redundant work and, in your reason, IDENTIFY THE SPECIFIC WASTEFUL CALLS: "
    "repeated `$metadata` schema rediscovery, trial-and-error `$filter` guessing (a failed call "
    "followed by retries that differ only in a fumbled filter — the `is_error` flags mark these), "
    "or fetching everything and filtering client-side when a server-side query would do.\n\n"
    "Score 1.0 for an efficient, well-chosen trajectory; lower toward 0.0 as redundant or failed "
    "calls increase. Your reason must explain WHICH calls were wasteful and WHY, and name the "
    "concrete fix (e.g. 'the second and third calls re-fetched $metadata; put the field names in "
    "the system prompt so the agent filters on the first call')."
)

llm_judge_results = {}

print("Running Strands Evals LLM-as-judge graders (via SAP GenAI Hub)...\n")

# --- OutputEvaluator: the task-success (outcome) judge. Fed the final answer + expected_response. ---
# include_inputs=True gives the judge the original question for context. This answers "was the
# agent right?"; the trajectory judge below answers "did it get there efficiently?".
output_judge = OutputEvaluator(rubric=OUTPUT_RUBRIC, model=model, include_inputs=True)

for scenario_name, run in local_runs.items():
    try:
        out = output_judge.evaluate(run["name_case"])[0]
        llm_judge_results[scenario_name] = [{
            "evaluatorId": "StrandsEvals.OutputEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        }]
        print(f"  {scenario_name}: OutputEvaluator={out.score:.2f} ({'success' if out.test_pass else 'fail'})")
    except Exception as e:
        print(f"  {scenario_name}: OutputEvaluator ERROR: {e}")
        llm_judge_results[scenario_name] = []

print()

# --- TrajectoryEvaluator: fed the DETAILED trajectory (call args), so it can name the waste. ---
# `trajectory_description` tells the judge what each tool does, so its reasoning about whether a
# call was necessary is grounded in the tool's actual capabilities (from extract_tools_description).
trajectory_judge = TrajectoryEvaluator(
    rubric=TRAJECTORY_RUBRIC,
    model=model,
    trajectory_description=TOOL_DESCRIPTIONS or None,
)

for scenario_name, run in local_runs.items():
    try:
        out = trajectory_judge.evaluate(run["detail_case"])[0]
        llm_judge_results.setdefault(scenario_name, []).append({
            "evaluatorId": "StrandsEvals.TrajectoryEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
        # Print the FULL reason — this is the "why is it inefficient" diagnosis we want to read.
        print(f"  {scenario_name}: TrajectoryEvaluator={out.score:.2f}")
        print(f"    {out.reason}\n")
    except Exception as e:
        print(f"  {scenario_name}: TrajectoryEvaluator ERROR: {e}\n")

# --- ToolSelectionAccuracyEvaluator: tool-level judge that needs a Session (OTEL spans). ---
# We use the Strands Evals Experiment + TracedHandler harness, which runs the agent and captures
# spans into a Session the tool-level judge can parse. This is heavier (re-runs the agent) and
# depends on local telemetry capture, so we guard it and fall back gracefully.
#
# We call `run_evaluations_async` and `await` it rather than the sync `run_evaluations`: the sync
# wrapper calls `asyncio.run()` internally, which raises inside a Jupyter kernel (it already has a
# running event loop). Notebook cells support top-level `await`, so the async variant is the right
# entry point here. `max_workers=1` keeps runs sequential (one SAP GenAI Hub call at a time),
# matching the sync wrapper's behaviour.
try:
    from strands_evals import Experiment, Case, eval_task, TracedHandler

    cases = [
        Case(
            name=s["name"],
            input=s["prompt"],
            expected_trajectory=s["expected_trajectory"],
        )
        for s in evaluation_scenarios
    ]

    @eval_task(TracedHandler())
    def warehouse_eval_task():
        # Fresh memory-free agent per case; the handler captures its spans into a Session.
        return create_warehouse_agent(with_memory=False)

    experiment = Experiment(
        cases=cases,
        evaluators=[ToolSelectionAccuracyEvaluator(model=model)],
    )
    reports = await experiment.run_evaluations_async(warehouse_eval_task, max_workers=1)

    # Each EvaluationReport carries parallel lists: cases[i] (a dict) and scores[i].
    # Fold the tool-selection scores into llm_judge_results, keyed by scenario name.
    for report in reports:
        for case_dict, score in zip(report.cases, report.scores):
            name = case_dict.get("name") if isinstance(case_dict, dict) else None
            if name is not None and score is not None:
                llm_judge_results.setdefault(name, []).append({
                    "evaluatorId": "StrandsEvals.ToolSelectionAccuracy",
                    "value": score,
                    "label": "justified" if score >= 0.5 else "unjustified",
                    "explanation": f"{report.evaluator_name} tool-selection score.",
                })
                print(f"  {name}: ToolSelectionAccuracy={score:.2f}")
    print("\nLLM-as-judge grading complete.")
except Exception as e:
    print(f"\n  ToolSelectionAccuracyEvaluator skipped (local trace capture unavailable): {e}")
    print("  OutputEvaluator and TrajectoryEvaluator results above still stand.")

### Local results

We combine the code-based and LLM-judge scores (plus token counts) into one table. The signal to
look for: a scenario that **reached an answer but scored low on the trajectory judge** — correct
but inefficient, exactly what an outcome-only grader misses. For any flagged scenario we print the
judge's explanation and the detailed trajectory so you can see the redundant calls.


In [ ]:
# Combine local Strands Evals scores (code-based + LLM-judge) into one table.
local_results = {}
for scenario in evaluation_scenarios:
    name = scenario["name"]
    local_results[name] = (
        tool_called_results.get(name, [])
        + llm_judge_results.get(name, [])
    )

LOCAL_EVALUATOR_IDS = [
    "StrandsEvals.ToolCalled",
    "StrandsEvals.OutputEvaluator",
    "StrandsEvals.TrajectoryEvaluator",
    "StrandsEvals.ToolSelectionAccuracy",
]

# TrajJudge score below this counts as an inefficient trajectory for the flagging logic.
TRAJECTORY_PASS_THRESHOLD = 0.5


def _local_short_name(eid):
    return {
        "StrandsEvals.ToolCalled": "ToolCalled",
        "StrandsEvals.OutputEvaluator": "TaskSuccess",
        "StrandsEvals.TrajectoryEvaluator": "TrajJudge",
        "StrandsEvals.ToolSelectionAccuracy": "ToolSelect",
    }.get(eid, eid.split(".")[-1][:12])


print("=" * 100)
print(" LOCAL EVALUATION — STRANDS EVALS (code-based + LLM-judge)")
print("=" * 100)

# Tokens is an informational efficiency signal (not a pass/fail grade): fewer tokens for the
# same correct answer is a cheaper, tighter trajectory.
header = f"{'Scenario':<25}"
for eid in LOCAL_EVALUATOR_IDS:
    header += f" {_local_short_name(eid):>14}"
header += f" {'Tokens':>10}"
print(header)
print("-" * 100)

local_scenario_scores = defaultdict(dict)
for name, results in local_results.items():
    row = f"{name:<25}"
    for eid in LOCAL_EVALUATOR_IDS:
        score = next((r.get("value", "-") for r in results if r.get("evaluatorId") == eid), "-")
        if isinstance(score, (int, float)):
            row += f" {score:>14.2f}"
            local_scenario_scores[name][eid] = score
        else:
            row += f" {str(score):>14}"
    tokens = local_runs.get(name, {}).get("tokens")
    row += f" {tokens:>10}" if isinstance(tokens, int) else f" {'-':>10}"
    print(row)
print("=" * 100)

# Highlight correct-but-inefficient: the task succeeded (TaskSuccess high) but the LLM trajectory
# judge scored the path low. The judge's reason names *why* it's wasteful, and we print the
# detailed trajectory beside it so you can see the redundant calls yourself. This is the gap an
# outcome-only score misses: a right answer reached the wrong way.
print("\nCorrect-but-inefficient check (task succeeded, but low trajectory-judge score):")
flagged = False
for name, results in local_results.items():
    output = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.OutputEvaluator"), {})
    called = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.ToolCalled"), {})
    traj = next((r for r in results if r.get("evaluatorId") == "StrandsEvals.TrajectoryEvaluator"), {})
    output_score = output.get("value")
    traj_score = traj.get("value")
    succeeded = (isinstance(output_score, (int, float)) and output_score >= 0.5) or called.get("tool_called")
    if succeeded and isinstance(traj_score, (int, float)) and traj_score < TRAJECTORY_PASS_THRESHOLD:
        flagged = True
        n_calls = called.get("n_tool_calls", "?")
        tokens = local_runs.get(name, {}).get("tokens")
        succ = f"{output_score:.2f}" if isinstance(output_score, (int, float)) else "n/a"
        print(f"\n  [!] {name}: succeeded (TaskSuccess {succ}) in {n_calls} tool call(s) / {tokens} tokens, "
              f"but TrajJudge scored {traj_score:.2f}")
        print(f"      Why: {traj.get('explanation', '')}")
        print("      Trajectory:")
        for i, call in enumerate(local_runs.get(name, {}).get("detailed", []), 1):
            print(f"        {i}. {call}")
if not flagged:
    print("  None flagged — every successful answer was reached via an efficient trajectory.")

### Apply a fix and re-test

The trajectory judge told us *what* was wasteful: the baseline agent rediscovers the schema at
runtime (repeated `$metadata`, trial-and-error `$filter`). `util/warehouse_agent.py` ships a fixed
prompt (`WAREHOUSE_SYSTEM_PROMPT_IMPROVED`) that gives the agent the entity name, field names, the
warehouse ID, and a worked `$filter`/`$select` example — so it can query correctly on the first
call.

We re-run the same scenarios against the fixed agent and grade it with the **same four Strands
Evals graders** used on the baseline (ToolCalled, TaskSuccess, TrajJudge, ToolSelect), plus tool
calls and tokens. The next cell then prints two tables: a **fixed-agent overview** on the same axes
as the baseline table above, and a **before → after** comparison across every metric — so you can
see the full picture, not just the trajectory score. This closes the evaluation loop:
measure → diagnose → fix → re-measure.


In [ ]:
# Re-run the scenarios against the FIXED agent (improved system prompt), then grade it with the
# SAME four Strands Evals graders used on the baseline (code-based ToolCalled + the three LLM
# judges), so the fix is measured on every metric — not just the trajectory score. Results are
# stored in `fixed_results`, mirroring `local_results`, so the next cell can print a full overview.
print("Re-running scenarios against the fixed agent (improved prompt)...\n")


def _make_fixed_agent():
    return create_warehouse_agent(with_memory=False, system_prompt=WAREHOUSE_SYSTEM_PROMPT_IMPROVED)


# evaluate_agent runs each scenario once and applies the code-based ToolCalled grader.
fixed_runs = evaluate_agent(_make_fixed_agent, evaluation_scenarios)

# Grade the fixed runs with the same LLM judges (OutputEvaluator, TrajectoryEvaluator) reused
# from the baseline cell, building per-scenario result lists shaped exactly like `local_results`.
print("\nGrading fixed answers and trajectories with the LLM judges...\n")
fixed_results = {}
for name, run in fixed_runs.items():
    results = [run["tool_called_result"]]  # StrandsEvals.ToolCalled (code-based)

    try:
        out = output_judge.evaluate(run["name_case"])[0]
        results.append({
            "evaluatorId": "StrandsEvals.OutputEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
    except Exception as e:
        print(f"  {name}: OutputEvaluator ERROR: {e}")

    try:
        out = trajectory_judge.evaluate(run["detail_case"])[0]
        run["traj_score"] = out.score
        run["traj_reason"] = out.reason
        results.append({
            "evaluatorId": "StrandsEvals.TrajectoryEvaluator",
            "value": out.score,
            "label": out.label or ("pass" if out.test_pass else "fail"),
            "explanation": out.reason,
        })
        print(f"  {name}: TaskSuccess + TrajJudge={out.score:.2f}")
    except Exception as e:
        run["traj_score"] = None
        run["traj_reason"] = f"error: {e}"
        print(f"  {name}: TrajectoryEvaluator ERROR: {e}")

    fixed_results[name] = results

# ToolSelectionAccuracy needs OTEL spans, so it runs through the same Experiment + TracedHandler
# harness used on the baseline — here with the improved-prompt agent. Guarded and optional.
print("\nGrading fixed tool selection (needs local trace capture)...\n")
try:
    from strands_evals import Experiment, Case, eval_task, TracedHandler

    fixed_cases = [
        Case(name=s["name"], input=s["prompt"], expected_trajectory=s["expected_trajectory"])
        for s in evaluation_scenarios
    ]

    @eval_task(TracedHandler())
    def fixed_warehouse_eval_task():
        return _make_fixed_agent()

    fixed_experiment = Experiment(
        cases=fixed_cases,
        evaluators=[ToolSelectionAccuracyEvaluator(model=model)],
    )
    fixed_reports = await fixed_experiment.run_evaluations_async(fixed_warehouse_eval_task, max_workers=1)

    for report in fixed_reports:
        for case_dict, score in zip(report.cases, report.scores):
            cname = case_dict.get("name") if isinstance(case_dict, dict) else None
            if cname is not None and score is not None:
                fixed_results.setdefault(cname, []).append({
                    "evaluatorId": "StrandsEvals.ToolSelectionAccuracy",
                    "value": score,
                    "label": "justified" if score >= 0.5 else "unjustified",
                    "explanation": f"{report.evaluator_name} tool-selection score.",
                })
                print(f"  {cname}: ToolSelectionAccuracy={score:.2f}")
    print("\nFixed-agent grading complete.")
except Exception as e:
    print(f"\n  ToolSelectionAccuracy skipped for fixed agent (local trace capture unavailable): {e}")
    print("  ToolCalled, TaskSuccess and TrajJudge results above still stand.")

In [ ]:
# Fixed-agent overview + full before/after comparison, across ALL local graders (not just the
# trajectory judge). The first table mirrors the baseline "LOCAL EVALUATION" table above so you
# can read the fixed agent on the same axes; the second puts every metric side by side, before→after.


def _score_for(results, eid):
    """Pull one evaluator's score out of a per-scenario result list (or None)."""
    return next((r.get("value") for r in results if r.get("evaluatorId") == eid), None)


def _fmt(score):
    return f"{score:.2f}" if isinstance(score, (int, float)) else "-"


# --- Fixed-agent overview (same columns as the baseline LOCAL EVALUATION table) ---
print("=" * 100)
print(" FIXED AGENT (improved prompt) — STRANDS EVALS (code-based + LLM-judge)")
print("=" * 100)
header = f"{'Scenario':<25}"
for eid in LOCAL_EVALUATOR_IDS:
    header += f" {_local_short_name(eid):>14}"
header += f" {'Tokens':>10}"
print(header)
print("-" * 100)
for name in (s["name"] for s in evaluation_scenarios):
    row = f"{name:<25}"
    for eid in LOCAL_EVALUATOR_IDS:
        row += f" {_fmt(_score_for(fixed_results.get(name, []), eid)):>14}"
    tokens = fixed_runs.get(name, {}).get("tokens")
    row += f" {tokens:>10}" if isinstance(tokens, int) else f" {'-':>10}"
    print(row)
print("=" * 100)

# --- Before → after, every metric (tool calls, tokens, and all four graders) ---
print("\n" + "=" * 118)
print(" BEFORE → AFTER (baseline vs. fixed agent) — all metrics")
print("=" * 118)
header = f"{'Scenario':<23} {'Calls':>11} {'Tokens':>17}"
for eid in LOCAL_EVALUATOR_IDS:
    header += f" {_local_short_name(eid) + ' b→a':>18}"
print(header)
print("-" * 118)
for name in (s["name"] for s in evaluation_scenarios):
    base, fix = local_runs.get(name, {}), fixed_runs.get(name, {})
    base_calls = base.get("tool_called_result", {}).get("n_tool_calls", "?")
    fix_calls = fix.get("tool_called_result", {}).get("n_tool_calls", "?")
    base_tok, fix_tok = base.get("tokens", "-"), fix.get("tokens", "-")
    row = (f"{name:<23} {f'{base_calls}→{fix_calls}':>11} "
           f"{f'{base_tok}→{fix_tok}':>17}")
    for eid in LOCAL_EVALUATOR_IDS:
        b = _fmt(_score_for(local_results.get(name, []), eid))
        a = _fmt(_score_for(fixed_results.get(name, []), eid))
        row += f" {f'{b}→{a}':>18}"
    print(row)
print("=" * 118)
print("\nReading it: TrajJudge is where the fix shows up most (fewer redundant OData calls, fewer "
      "tokens), while TaskSuccess stays high — the improved prompt made the agent *faster*, "
      "not *more correct*. That is the whole point of grading trajectory alongside outcome.")

---

## Stage 2 — Deployed evaluation with AgentCore

Stage 1 answered *"is the logic good?"* and produced a fix. But that fix only lives in this
notebook — the deployed runtime still runs the baseline prompt. Stage 2 answers the second
question: **did the fix ship, and how does the real thing behave under production infrastructure?**

So the flow here is deploy-then-verify:

1. **Ensure memory** — a memory resource that **Lab 9 owns** (idempotent create-or-find in
   section 1's `ensure_memory`), so Stage 2 doesn't depend on Lab 8 having run.
2. **Redeploy the improved prompt in-place** to the same Lab 7 runtime, memory-enabled.
3. **Grade the improved runtime** with **AgentCore Evaluations**, which scores it from its
   **OTEL traces** rather than an in-process result: the deployed agent emits spans to CloudWatch,
   we invoke it per scenario and wait (~180s) for ingestion, then `EvaluationClient.run()` reads
   those spans and scores them with built-in and custom LLM-as-judge evaluators.
4. **Build a scorecard + cross-stage tie-back** — a parity check (Correctness / ToolSelectionAccuracy /
   OpsQuality: did quality hold in prod?) plus prod-only latency and cost, framed as a before→after
   against Stage 1 (local naive → prod improved).

This is what you run against the *actual* production agent, and the same mechanism you'd wire into
continuous online evaluation for monitoring.

### Configure evaluation infrastructure

Set up the AgentCore Runtime client and helper functions for invoking the deployed agent and
waiting for OTEL span ingestion.


In [ ]:
from datetime import timedelta

# AgentCore Runtime client for invoking the deployed agent
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

# Derive the CloudWatch log group where OTEL spans land
CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

# Ingestion delay — time to wait for OTEL spans to arrive in CloudWatch.
INGESTION_DELAY = 180


def invoke_with_retry(client, agent_arn, session_id, prompt, max_retries=3, wait=30):
    """Invoke agent runtime with retry for cold start 500 errors.

    Returns (response, session_id) — the session id it actually SUCCEEDED with. On a cold
    start we mutate the session id per retry (`..._r1`, `..._r2`), so the successful
    invocation may run under a different id than the caller passed in; the caller needs that
    effective id to locate the right OTEL spans later.
    """
    for attempt in range(max_retries):
        try:
            response = client.invoke_agent_runtime(
                agentRuntimeArn=agent_arn,
                qualifier="DEFAULT",
                runtimeSessionId=session_id,
                payload=json.dumps({"prompt": prompt}).encode("utf-8"),
            )
            return response, session_id
        except client.exceptions.RuntimeClientError as e:
            if attempt < max_retries - 1:
                print(f"  Runtime error (attempt {attempt + 1}/{max_retries}), retrying in {wait}s (likely cold start)...")
                time.sleep(wait)
                session_id = f"{session_id}_r{attempt + 1}"
            else:
                raise e
    return None, session_id


def invoke_with_timing(client, agent_arn, session_id, prompt):
    """Wrap invoke_with_retry to record wall-clock latency and a cold-start flag.

    Returns (response, timing) where timing carries the EFFECTIVE session id under which the
    invocation succeeded (`timing["session_id"]`). On the happy path that is the original id;
    on a cold start it is whatever invoke_with_retry ended up succeeding with, so downstream
    span/cost lookups query the right session.
    """
    start = time.time()
    cold = False
    effective_session_id = session_id
    try:
        response = client.invoke_agent_runtime(
            agentRuntimeArn=agent_arn, qualifier="DEFAULT",
            runtimeSessionId=session_id,
            payload=json.dumps({"prompt": prompt}).encode("utf-8"),
        )
    except client.exceptions.RuntimeClientError:
        cold = True
        response, effective_session_id = invoke_with_retry(client, agent_arn, session_id, prompt)
    return response, {
        "latency_s": time.time() - start,
        "cold_start": cold,
        "session_id": effective_session_id,
    }


print(f"Agent ID: {AGENT_ID}")
print(f"Agent ARN: {AGENT_ARN}")
print(f"CloudWatch Log Group: {CW_LOG_GROUP}")
print(f"Ingestion delay: {INGESTION_DELAY}s")

### Create the custom evaluator

We register a domain-specific LLM-as-a-judge evaluator in the AgentCore control plane. This complements the built-in evaluators with SAP warehouse-specific scoring criteria that generic evaluators cannot assess.

**WarehouseOperationalQuality** (TRACE-level): Evaluates whether the agent's response is operationally useful for a warehouse manager — does it provide actionable inventory insights, use correct product codes, and present data in a way that supports procurement decisions?

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

_SUFFIX = uuid.uuid4().hex[:8]

# TODO: Use the inference profile matching your region (us.* for us-east-1, eu.* for eu-central-1)
JUDGE_MODEL_ID = "us.amazon.nova-pro-v1:0"

# Custom TRACE-level evaluator: Warehouse Operational Quality
print("Creating WarehouseOperationalQuality evaluator (TRACE-level)...")
warehouse_quality_response = agentcore_control.create_evaluator(
    evaluatorName=f"WarehouseOperationalQuality_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "You are a warehouse operations expert evaluating an AI assistant that queries "
                "SAP S/4HANA warehouse APIs for inventory management.\n\n"
                "Conversation context: {context}\n"
                "Agent response: {assistant_turn}\n"
                "Expected behavior: {expected_response}\n\n"
                "Evaluate the OPERATIONAL QUALITY of the response for a warehouse manager. Score based on:\n"
                "1. Does the response contain specific, quantitative inventory data (not vague statements)?\n"
                "2. Are SAP product codes (WM-AN01, WM-AN02, WM-AN03, WM-AN04) used correctly?\n"
                "3. Is the data presented in a way that supports immediate operational decisions "
                "(e.g., reorder recommendations, fulfillment feasibility, capacity utilization)?\n"
                "4. Does the response avoid hallucinating inventory numbers when API data is unavailable?\n\n"
                "Important: If the agent successfully queried the API and returned real data with "
                "correct product codes and actionable insights, score 1.0 even if formatting differs "
                "from the expected response.\n\n"
                "You MUST respond with EXACTLY one of these scores:\n"
                "- 0.0 if the response lacks inventory data, hallucinates numbers, or is not actionable\n"
                "- 0.5 if the response has some useful data but is missing key operational context\n"
                "- 1.0 if the response provides accurate, actionable warehouse intelligence\n\n"
                "Respond with only the numeric score (0.0, 0.5, or 1.0) on the first line, "
                "followed by a one-sentence explanation on the next line."
            ),
            "ratingScale": {
                "numerical": [
                    {"value": 0.0, "label": "not_actionable", "definition": "Response lacks data, hallucinates numbers, or provides no operational value."},
                    {"value": 0.5, "label": "partially_useful", "definition": "Some useful data present but missing key operational context for decisions."},
                    {"value": 1.0, "label": "operationally_excellent", "definition": "Accurate, specific, and actionable warehouse intelligence."},
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": JUDGE_MODEL_ID,
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_EVALUATOR_ID = warehouse_quality_response["evaluatorId"]
print(f"  Created: {CUSTOM_EVALUATOR_ID}")
print(f"\nCustom evaluator registered in AgentCore control plane.")

### Ship the Stage-1 fix: redeploy the improved runtime

Stage 1 found the baseline agent correct-but-inefficient and produced a fix
(`WAREHOUSE_SYSTEM_PROMPT_IMPROVED`). But **that fix never reached production** — the deployed
runtime still runs the baseline prompt. Before we can verify it in prod, we have to ship it. The
next cells redeploy the improved prompt **in-place** to the same Lab 7 runtime (reusing
`AGENT_NAME` with `auto_update_on_conflict=True`, so `AGENT_ID` / `AGENT_ARN` stay valid),
memory-enabled via the resource `ensure_memory` created in section 1.

Because Lab 9 owns that memory resource, Stage 2 does **not** depend on Lab 8 having run — this
notebook stands on its own.

> **Time & cost:** the redeploy cell triggers a real CodeBuild and takes **~4 min**; the full
> Stage 2 (redeploy → invoke per scenario → ~180s OTEL ingestion wait → grade) runs **~9–12 min
> total**. It also makes live Bedrock/AgentCore calls, so it incurs cost.


In [ ]:
%%writefile warehouse_agent_agentcore.py
# Lab 09 Stage 2: improved prompt (known-schema hint) — redeployed to verify the Stage-1 fix in prod.
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.runtime.context import RequestContext
from util.strands_bedrock_sap_genai_hub import SAPGenAIHubModel
from strands import Agent, tool
from strands.hooks import (
    AfterInvocationEvent,
    MessageAddedEvent,
    HookProvider,
    HookRegistry,
)
import os
from pathlib import Path
import yaml
from util.odata_tool import odata_caller

from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole, RetrievalConfig
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

# Initialize AgentCore app
app = BedrockAgentCoreApp()

# Initialize the SAPGenAIHubModel for AgentCore deployment
model = SAPGenAIHubModel(
    model_id="anthropic--claude-4.5-sonnet",
    # Alternative models:
    # model_id="amazon--nova-pro",
    # model_id="amazon--nova-lite",
)

# --- Memory configuration (injected as container env vars at launch) ---
REGION = os.environ.get("AWS_REGION", "us-east-1")
MEMORY_ID = os.environ.get("MEMORY_ID") or os.environ.get("BEDROCK_AGENTCORE_MEMORY_ID")  # section 2 resource (toolkit-managed memory injects BEDROCK_AGENTCORE_MEMORY_ID)
ACTOR_ID = os.environ.get("ACTOR_ID", "warehouse_manager_001")

_session_manager = MemorySessionManager(memory_id=MEMORY_ID, region_name=REGION) if MEMORY_ID else None


class WarehouseMemoryHooks(HookProvider):
    """Memory hooks for the deployed warehouse agent, scoped to one MemorySession."""

    def __init__(self, warehouse_session: MemorySession):
        self.warehouse_session = warehouse_session
        self.retrieval_config = {
            "warehouse/{actorId}/facts/": RetrievalConfig(top_k=5, relevance_score=0.2),
            "warehouse/{actorId}/preferences/": RetrievalConfig(top_k=3, relevance_score=0.3),
        }

    def retrieve_warehouse_context(self, event: MessageAddedEvent):
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            memory_context_parts = []
            try:
                recent_turns = self.warehouse_session.get_last_k_turns(k=5)
                if recent_turns:
                    history_lines = []
                    for turn in recent_turns:
                        for message in turn:
                            role = message.get("role", "unknown")
                            content = message.get("content", {}).get("text", "")
                            history_lines.append(f"{role}: {content}")
                    memory_context_parts.append("Recent Conversation:\n" + "\n".join(history_lines))
            except Exception as e:
                print(f"  [Memory] Short-term load failed: {e}")
            try:
                for namespace_template, config in self.retrieval_config.items():
                    resolved_namespace = namespace_template.format(actorId=self.warehouse_session._actor_id)
                    memories = self.warehouse_session.search_long_term_memories(
                        query=user_query, namespace_prefix=resolved_namespace, top_k=config.top_k,
                    )
                    filtered = [m for m in memories if m.get("score", 0) >= config.relevance_score]
                    if filtered:
                        lines = [f"- {m['content']['text']}" for m in filtered[:5]]
                        label = "Facts" if "facts" in namespace_template else "Preferences"
                        memory_context_parts.append(f"Known {label}:\n" + "\n".join(lines))
            except Exception as e:
                print(f"  [Memory] Long-term load failed: {e}")
            if memory_context_parts:
                context_block = "\n\n".join(memory_context_parts)
                event.agent.system_prompt += (
                    f"\n\n--- MEMORY CONTEXT ---\n{context_block}\n"
                    "Use this context to personalize responses. Do not ask for information "
                    "you already know from memory.\n--- END MEMORY CONTEXT ---"
                )
                print(f"  [Memory] Injected {len(memory_context_parts)} memory sections")

    def save_warehouse_interaction(self, event: AfterInvocationEvent):
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                user_query = None
                agent_response = None
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        if msg["content"] and msg["content"][0].get("text"):
                            agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_query and "toolResult" not in msg["content"][0]:
                        user_query = msg["content"][0]["text"]
                        break
                if user_query and agent_response:
                    interaction_messages = [
                        ConversationalMessage(user_query, MessageRole.USER),
                        ConversationalMessage(agent_response, MessageRole.ASSISTANT),
                    ]
                    result = self.warehouse_session.add_turns(interaction_messages)
                    print(f"  [Memory] Saved interaction - Event ID: {result['eventId']}")
        except Exception as e:
            print(f"  [Memory] Save failed: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(MessageAddedEvent, self.retrieve_warehouse_context)
        registry.add_callback(AfterInvocationEvent, self.save_warehouse_interaction)


# Load YAML OpenAPI files from directory
def load_openapi_specs(path="./assets/knowledgebase"):
    specs = []
    for file in Path(path).glob("*.y*ml"):
        with open(file, "r") as f:
            try:
                data = yaml.safe_load(f)
                servers = data.get("servers", [])
                base_urls = []
                for s in servers:
                    url = s.get("url")
                    desc = s.get("description", "")
                    if url:
                        base_urls.append({"url": url, "description": desc})
                summary = {
                    "file": file.name,
                    "title": data.get("info", {}).get("title"),
                    "description": data.get("info", {}).get("description"),
                    "paths": list(data.get("paths", {}).keys()),
                    "base_urls": base_urls or [{"url": "/", "description": "Default"}]
                }
                specs.append(summary)
            except Exception as e:
                print(f"Error parsing {file}: {e}")
    return specs

def specs_to_prompt_string(specs):
    prompt_parts = []
    for spec in specs:
        base_url_str = ", ".join(
            f"{url_info['url']} ({url_info['description']})" for url_info in spec['base_urls']
        )
        paths_str = ", ".join(spec['paths'])
        part = (
            f"API Spec: {spec['file']}\n"
            f"Title: {spec['title']}\n"
            f"Description: {spec['description']}\n"
            f"Base URLs: {base_url_str}\n"
            f"Endpoints: {paths_str}\n"
        )
        prompt_parts.append(part)
    return "\n---\n".join(prompt_parts)

# Create API specs for selector agent
specs = load_openapi_specs()
specs_prompt_string = specs_to_prompt_string(specs)

SELECTOR_SYSTEM_PROMPT = f"""
You are an API selection subagent.
Given a user query and the following list of OpenAPI specs, each with a title, description, base URLs, and available endpoints:

{specs_prompt_string}

Your task is to identify which API and specific endpoint is most appropriate to fulfill the user query.
Provide clear reasoning for your choice, referencing the API spec details provided.
If no suitable API is found, explain why.

Make sure to reply with the sandbox BASE URL.

"""

# Create selector agent
selector_agent = Agent(
    system_prompt=SELECTOR_SYSTEM_PROMPT,
    model=model
)

@tool
def SelectorAPIAgentAsATool(query: str) -> str:
    """
    This analyzes available OpenAPI specs and selects the most appropriate
    API and endpoint based on user queries.

    Args:
        query: User query describing what they want to accomplish

    Returns:
        Agent response with API selection recommendation if successful,
        error message if initialization fails
    """
    return selector_agent(query).message

# Define the warehouse agent system prompt (identical to Lab 7)
warehouse_agent_prompt = """
You are an expert Warehouse Operations Manager for GlobalTech Manufacturing's Distribution Center (Warehouse 1750).
You have access to real-time SAP warehouse data through dynamic OData API exploration capabilities.

CORE CAPABILITIES:
1. API Structure Exploration: Dynamically discover available data fields and entities
2. Product Discovery: Find available products without hardcoded assumptions
3. Dynamic Querying: Construct intelligent OData queries based on user needs

APPROACH TO PROBLEM SOLVING:
- You must start by using the SelectorAPIAgentAsATool to help you determine which OData API to call
- Next use the $metadata endpoint that to understand the API that SelectorAPIAgentAsATool provides
- Use dynamic queries to discover information rather than making assumptions
- Leverage OData filtering, sorting, and selection to get precise answers
- You may need to do this multiple times. Always write what URL have constructed so user is informed.

PRODUCT KNOWLEDGE (can be expanded through discovery):
- WM-AN01: Advanced Sensors (high-precision electronic components)
- WM-AN02: Control Units (critical automation hardware)
- WM-AN03: Power Modules (electrical power management systems)
- WM-AN04: Communication Devices (networking and connectivity hardware)

COMMUNICATION STYLE:
- Be professional but conversational and succinct
- Explain your discovery process when exploring new data
- Provide specific, actionable insights with quantitative data
- If you know the user's preferences from memory, apply them without asking
- Do not use emojis

When users ask questions:
1. First determine what data you need to answer the question
3. Feel free to use the odata_caller tool as many times as needed to get the right information.
3. Construct appropriate OData queries to get the specific information needed
4. Analyze the results and provide comprehensive, intelligent responses

Example for using the odata_caller tool:
```python
        odata_caller(
            base_url="",
            endpoint="WarehouseStockProducts",
            operation="get",
            odata_params={"$filter": "Product eq 'WM-AN02'"},
            auth_type="api_key",
            auth_env_var="SAP_S4HANA_PUBLIC_CLOUD_KEY"
        )
        ```

Focus on SAP S/4 HANA OData endpoints, warehouse management APIs, and supply chain operations.

Available tools:
- odata_caller: Universal OData tool for SAP API interactions with built-in authentication and query parameter support
- auth_token: Use os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY") to get the API key
Example below:

# For SAP OData calls, use consistent headers
headers = {
    "APIKey": os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY"),
    "Accept": "application/json", OR "application/xml" choose as needed
    "DataServiceVersion": "2.0"
}

The odata_caller tool handles SAP-specific authentication automatically and provides comprehensive error handling and response formatting.

KNOWN SCHEMA — use it directly for warehouse physical-stock queries. Do NOT call the
SelectorAPIAgentAsATool and do NOT read $metadata for these; you already know the API:
- base_url (service root): "https://sandbox.api.sap.com/s4hanacloud/sap/opu/odata4/sap/api_whse_physstockprod/srvd_a2x/sap/whsephysicalstockproducts/0001"
- endpoint (entity set): "WarehousePhysicalStockProducts"
- Warehouse ID: EWMWarehouse = "1750"
- Key fields:
  - Product                       (e.g. "WM-AN02")
  - EWMWarehouse                  (warehouse number, "1750")
  - EWMStorageBin                 (storage bin)
  - EWMStorageType                (storage type)
  - EWMStockType                  (stock type)
  - EWMStockQuantityInBaseUnit    (on-hand quantity, numeric)
  - EWMStockQuantityBaseUnit      (unit of measure)

Query the entity set on the FIRST call with a server-side $filter and $select — never fetch
everything and filter client-side. Worked example for a single product:
  odata_caller(
    base_url="https://sandbox.api.sap.com/s4hanacloud/sap/opu/odata4/sap/api_whse_physstockprod/srvd_a2x/sap/whsephysicalstockproducts/0001",
    endpoint="WarehousePhysicalStockProducts",
    operation="get",
    odata_params={
      "$filter": "Product eq 'WM-AN02' and EWMWarehouse eq '1750'",
      "$select": "Product,EWMStockQuantityInBaseUnit,EWMStockQuantityBaseUnit,EWMStorageBin",
    },
    auth_type="api_key",
    auth_env_var="SAP_S4HANA_PUBLIC_CLOUD_KEY",
  )
For all products, filter on EWMWarehouse eq '1750' alone; sort with $orderby when you need
low-stock items (e.g. "$orderby": "EWMStockQuantityInBaseUnit asc").
"""


def build_warehouse_agent(session_id: str | None):
    """Create the Lab 7 warehouse agent, attaching memory hooks when a session is available."""
    hooks = []
    if _session_manager and session_id:
        try:
            warehouse_session = _session_manager.create_memory_session(
                actor_id=ACTOR_ID, session_id=session_id
            )
            hooks.append(WarehouseMemoryHooks(warehouse_session))
        except Exception as e:
            print(f"  [Memory] Session init failed, running without memory: {e}")

    return Agent(
        model=model,
        tools=[SelectorAPIAgentAsATool, odata_caller],
        system_prompt=warehouse_agent_prompt,
        hooks=hooks,
    )


# AgentCore entrypoint function. The second parameter MUST be named `context` for the
# runtime to inject the RequestContext (its .session_id is the runtimeSessionId used to
# scope memory to this invocation).
@app.entrypoint
def warehouse_agent_entrypoint(payload, context: RequestContext):
    """Invoke the memory-enhanced warehouse agent with a payload."""
    user_input = payload.get("prompt")
    session_id = getattr(context, "session_id", None)
    print("User input:", user_input, "| session:", session_id)
    warehouse_agent = build_warehouse_agent(session_id)
    response = warehouse_agent(user_input)
    return response.message

if __name__ == "__main__":
    app.run()

In [ ]:
# Redeploy the memory-enhanced agent to the SAME AgentCore Runtime from Lab 7.
# Reusing the Lab 7 agent name + auto_update_on_conflict=True updates in place, so
# AGENT_ID / AGENT_ARN (loaded in section 1) stay valid for the Lab 9 evaluation.
import getpass
from bedrock_agentcore_starter_toolkit import Runtime
from dotenv import load_dotenv

load_dotenv()

# AGENT_NAME / AGENT_ID were loaded from lab7_deployment.json in section 1.
print(f"Updating existing AgentCore Runtime: {AGENT_NAME} ({AGENT_ID})")

# The agent manages its own memory: the entrypoint (build_warehouse_agent) reads the
# section-2 memory resource from the MEMORY_ID env var and wires up MemorySessionManager +
# WarehouseMemoryHooks itself. MEMORY_ID / ACTOR_ID are injected at launch() below, so we do
# NOT enable the toolkit's managed-memory feature here (that would provision a second,
# redundant memory resource). This keeps configure() compatible with the pinned
# bedrock-agentcore-starter-toolkit 0.1.14, whose configure() has no memory_mode parameter.
agentcore_runtime = Runtime()
configure_response = agentcore_runtime.configure(
    entrypoint="warehouse_agent_agentcore.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=AGENT_NAME,
)

if not memory_id:
    raise SystemExit("memory_id is not set — run section 2 (Create AgentCore Memory Resource) first.")

print(f"AgentCore configuration completed. The agent will bind to memory resource "
      f"{memory_id} via the MEMORY_ID env var at launch. Modifying the generated Dockerfile next...")
configure_response

In [ ]:
# Re-apply the Lab 7 Dockerfile modifications so the container has util/, assets/, and
# the SAP GenAI Hub credentials (config.json -> /app/.aicore via AICORE_HOME).

# Materialize config.json from ~/.aicore/config.json into the build context. This file is
# gitignored and transient (Lab 7 writes it the same way), so it may be absent when Lab 8
# runs on its own — the Dockerfile's `COPY config.json` below fails the CodeBuild step
# without it.
config_path = os.path.expanduser("~/.aicore/config.json")
if not os.path.exists(config_path):
    raise SystemExit(
        f"{config_path} not found — run notebook 00 to configure SAP AI Core credentials first."
    )
with open(config_path, "r") as f:
    _aicore_config = json.load(f)
with open("config.json", "w") as f:
    json.dump(_aicore_config, f, indent=2)
print("SAP AI Core config.json written into the build context.")

with open("Dockerfile", "r") as f:
    dockerfile_content = f.read()

lines = dockerfile_content.split("\n")
cmd_index = next((i for i, line in enumerate(lines) if line.strip().startswith("CMD")), -1)

if cmd_index > -1 and "AICORE_HOME" not in dockerfile_content:
    config_lines = [
        "",
        "# Copy util directory (required for SAP GenAI Hub model and OData tool)",
        "COPY util/ ./util/",
        "",
        "# Copy assets directory (required for OpenAPI knowledgebase)",
        "COPY assets/ ./assets/",
        "",
        "# Copy the config.json from your local machine",
        "COPY config.json /app/.aicore/config.json",
        "",
        "# Set AICORE_HOME environment variable for SAP GenAI Hub SDK",
        "ENV AICORE_HOME=/app/.aicore",
        "",
    ]
    modified_lines = lines[:cmd_index] + config_lines + lines[cmd_index:]
    with open("Dockerfile", "w") as f:
        f.write("\n".join(modified_lines))
    print("Dockerfile modified with util/, assets/, and SAP GenAI Hub configuration.")
elif "AICORE_HOME" in dockerfile_content:
    print("Dockerfile already contains SAP GenAI Hub configuration — skipping.")
else:
    print("Could not find CMD instruction in Dockerfile.")

In [ ]:
# Launch the update. We pass MEMORY_ID and ACTOR_ID so the deployed agent binds to the
# AgentCore Memory resource created in section 2, plus the SAP API key for OData calls.
sap_api_key = os.getenv("SAP_S4HANA_PUBLIC_CLOUD_KEY")
if not sap_api_key:
    sap_api_key = getpass.getpass("Please enter your SAP_S4HANA_PUBLIC_CLOUD_KEY: ")

if not memory_id:
    raise SystemExit("memory_id is not set — run section 2 (Create AgentCore Memory Resource) first.")

launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "SAP_S4HANA_PUBLIC_CLOUD_KEY": sap_api_key,
        "MEMORY_ID": memory_id,
        "ACTOR_ID": ACTOR_ID,
    },
)
print("Memory-enhanced deployment initiated on the existing runtime.")
launch_result

In [ ]:
# Wait for the updated runtime to reach READY before evaluating it in Lab 9.
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

print(f"Initial status: {status}")
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)

print(f"\nFinal status: {status}")

# Refuse to grade a half-deployed runtime. If the redeploy did not reach READY, the runtime
# still runs the STALE baseline prompt — grading it would report the old agent as "improved"
# and silently invalidate the whole parity story. Abort Stage 2 instead.
if status != "READY":
    raise SystemExit(
        f"Redeploy did not reach READY (final status: {status}). Stage 2 aborted — "
        "refusing to grade a half-deployed runtime."
    )

print(f"The deployed agent (AGENT_ID={AGENT_ID}) is now memory-enhanced and ready for Lab 9 evaluation.")
status

### Invoke the deployed agent for each scenario

We invoke the deployed agent (from Lab 7) for each evaluation scenario. Each invocation gets a unique `runtimeSessionId` so the evaluator can locate its spans independently.

In [ ]:
# Invoke the deployed agent for each scenario with timing, and collect per-scenario sessions.


def parse_agent_response(response_body: str) -> str:
    """Parse invoke_agent_runtime response into plain text.

    The response is a JSON object: {"role": "assistant", "content": [{"text": "..."}], "metadata": {...}}
    It may arrive as a single blob or as multiple newline-delimited chunks that concatenate into one object.
    """
    text_parts = []

    # Try parsing as a single JSON object (most common)
    try:
        data = json.loads(response_body)
        if isinstance(data, dict) and "content" in data:
            for block in data["content"]:
                if isinstance(block, dict) and "text" in block:
                    text_parts.append(block["text"])
            return "".join(text_parts)
    except json.JSONDecodeError:
        pass

    # Fall back: response may be multiple concatenated JSON chunks (chunked transfer)
    try:
        combined = "".join(response_body.strip().split("\n"))
        data = json.loads(combined)
        if isinstance(data, dict) and "content" in data:
            for block in data["content"]:
                if isinstance(block, dict) and "text" in block:
                    text_parts.append(block["text"])
            return "".join(text_parts)
    except json.JSONDecodeError:
        pass

    # Last resort: return raw truncated
    return response_body[:500]


def run_prod_round(label, scenarios):
    """Invoke each scenario against the deployed runtime with timing, wait for
    span ingestion, and return per-scenario session dicts (timing included)."""
    sessions = []
    print(f"[{label}] Invoking deployed agent for {len(scenarios)} scenarios...\n")
    for scenario in scenarios:
        session_id = f"eval_{scenario['name']}_{uuid.uuid4().hex}"  # >= 33 chars
        try:
            response, timing = invoke_with_timing(agentcore_client, AGENT_ARN, session_id, scenario["prompt"])
            response_body = response["response"].read().decode("utf-8")
            agent_text = parse_agent_response(response_body)
            # Store the EFFECTIVE session id the invocation succeeded under (timing["session_id"]),
            # not the pre-generated one: a cold-start retry mutates the id, and grade_sessions /
            # read_span_tokens must query the id whose spans actually exist.
            sessions.append({
                "scenario_name": scenario["name"], "session_id": timing["session_id"],
                "prompt": scenario["prompt"], "response": agent_text[:500],
                "expected_response": scenario["expected_response"],
                "expected_trajectory": scenario["expected_trajectory"],
                "assertions": scenario["assertions"],
                "latency_s": timing["latency_s"], "cold_start": timing["cold_start"],
            })
            print(f"  [{scenario['name']}] {timing['latency_s']:.1f}s"
                  f"{' (cold start)' if timing['cold_start'] else ''}")
        except Exception as e:
            print(f"  [{scenario['name']}] ERROR: {e}")
    print(f"\n[{label}] Waiting {INGESTION_DELAY}s for CloudWatch span ingestion...")
    time.sleep(INGESTION_DELAY)
    return sessions

In [ ]:
import re

logs_client = boto3.client("logs", region_name=REGION)


def read_span_tokens(session_id):
    """Sum input/output tokens from the session's OTEL spans in CloudWatch.
    Returns (None, None) if spans/usage are not found — never fabricates."""
    try:
        events = logs_client.filter_log_events(
            logGroupName=CW_LOG_GROUP,
            filterPattern=f'"{session_id}"',
            limit=200,
        ).get("events", [])
    except Exception as e:
        print(f"  [cost] log read failed: {e}")
        return None, None
    inp = out = 0
    found = False
    for ev in events:
        msg = ev.get("message", "")
        for key, acc in (("gen_ai.usage.input_tokens", "in"), ("gen_ai.usage.output_tokens", "out")):
            # Sum ALL occurrences of the key: the OTEL exporter can batch several spans
            # (selector sub-agent + warehouse agent + multiple LLM turns) into one CloudWatch
            # event, so msg.find (first hit only) would under-count. finditer catches every one.
            for m in re.finditer(re.escape(key) + r"[^0-9]*(\d+)", msg):
                # Only flip `found` once an integer actually parses — a key present with no
                # parseable number must NOT turn (None, None) into a fabricated (0, 0).
                found = True
                if acc == "in":
                    inp += int(m.group(1))
                else:
                    out += int(m.group(1))
    return (inp, out) if found else (None, None)

### Evaluate with built-in evaluators

We use `EvaluationClient.run()` to score each session with AgentCore's built-in evaluators:

| Evaluator | Level | Needs Ground Truth | What it measures |
|-----------|-------|-------------------|-----------------|
| `Builtin.Correctness` | TRACE | `expected_response` | Factual accuracy of the response |
| `Builtin.Helpfulness` | TRACE | None | How useful/valuable the response is |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` | Whether the agent completed the user's goal |
| `Builtin.ToolSelectionAccuracy` | SESSION | None | Whether the agent chose the right tools |

In [ ]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

ec = EvaluationClient(region_name=REGION)

# Pre-populate the evaluator level cache (required for the SDK to route correctly)
ec._evaluator_level_cache.update({
    "Builtin.Correctness": "TRACE",
    "Builtin.Helpfulness": "TRACE",
    "Builtin.GoalSuccessRate": "SESSION",
    "Builtin.ToolSelectionAccuracy": "SESSION",
})

BUILTIN_EVALUATOR_IDS = [
    "Builtin.Correctness",
    "Builtin.Helpfulness",
    "Builtin.GoalSuccessRate",
    "Builtin.ToolSelectionAccuracy",
]

# The custom WarehouseOperationalQuality evaluator (registered above) is TRACE-level too, so
# grade_sessions can run it in the same pass as the built-ins.
ec._evaluator_level_cache[CUSTOM_EVALUATOR_ID] = "TRACE"


def grade_sessions(sessions):
    """Grade prod sessions with the built-in + custom AgentCore evaluators.

    Runs BUILTIN_EVALUATOR_IDS then the custom WarehouseOperationalQuality evaluator on each
    session's OTEL traces and returns {scenario_name: [built-in results..., custom results...]}
    — the combined shape the scorecard and display consume. Any evaluator error falls back to
    [] for that group so one bad session never aborts the round.
    """
    graded = {}
    print(f"Grading {len(sessions)} session(s) with AgentCore built-in + custom evaluators...\n")
    for session in sessions:
        name = session["scenario_name"]
        print(f"  Evaluating: {name} (session: {session['session_id']})")

        reference_inputs = ReferenceInputs(
            expected_response=session["expected_response"],
            expected_trajectory=session["expected_trajectory"],
            assertions=[session["assertions"]],
        )

        try:
            builtin = ec.run(
                evaluator_ids=BUILTIN_EVALUATOR_IDS,
                agent_id=AGENT_ID,
                session_id=session["session_id"],
                look_back_time=timedelta(hours=1),
                reference_inputs=reference_inputs,
            )
            for r in builtin:
                print(f"    {r.get('evaluatorId', 'unknown')}: {r.get('value', 'N/A')} ({r.get('label', '')})")
        except Exception as e:
            print(f"    built-in ERROR: {e}")
            builtin = []

        try:
            custom = ec.run(
                evaluator_ids=[CUSTOM_EVALUATOR_ID],
                agent_id=AGENT_ID,
                session_id=session["session_id"],
                look_back_time=timedelta(hours=1),
                reference_inputs=reference_inputs,
            )
            for r in custom:
                label = r.get("label", "")
                print(f"    OpsQuality: {r.get('value', 'N/A')} ({label})")
                if r.get("explanation"):
                    print(f"      {r['explanation'][:120]}")
        except Exception as e:
            print(f"    custom ERROR: {e}")
            custom = []

        graded[name] = builtin + custom
        print()

    print("Grading complete.")
    return graded


### Aggregate one round into a scorecard

`grade_sessions` (above) runs the built-in **and** the custom **WarehouseOperationalQuality**
evaluator on every session — the custom judge scores whether the agent provides actionable
warehouse intelligence, something the generic built-ins cannot assess. The next cell adds
`build_scorecard`, which folds one graded round into a flat metric dict (Correctness,
ToolSelectionAccuracy, OpsQuality, plus prod-only latency and cost) ready for `render_scorecard`.


In [ ]:
# Aggregate one graded prod round into a flat scorecard dict for render_scorecard.
from util.deployed_eval import tokens_to_cost, summarize_timing


def build_scorecard(sessions, graded):
    """Aggregate one prod round into a flat metric dict for render_scorecard."""
    def avg_metric(eid):
        vals = [
            r.get("value") for s in sessions
            for r in graded.get(s["scenario_name"], [])
            if r.get("evaluatorId") == eid and isinstance(r.get("value"), (int, float))
        ]
        return sum(vals) / len(vals) if vals else None

    # Cost is a prod-only signal read from the OTEL spans. read_span_tokens can return (0, 0)
    # when a token key is present but no integer parses — treat 0 the same as None (cost
    # unavailable) so we never fabricate a $0.00 cost.
    costs = []
    for s in sessions:
        inp, out = read_span_tokens(s["session_id"])
        if inp and out:  # 0 or None -> skip
            c = tokens_to_cost(inp, out, model.get_config()["model_id"])
            if c is not None:
                costs.append(c)

    timing = summarize_timing([{"latency_s": s["latency_s"], "cold_start": s["cold_start"]} for s in sessions])
    return {
        "Correctness": avg_metric("Builtin.Correctness"),
        "ToolSelectionAccuracy": avg_metric("Builtin.ToolSelectionAccuracy"),
        "OpsQuality": avg_metric(CUSTOM_EVALUATOR_ID),
        "latency_s": timing["mean_s"],
        "cost_usd": (sum(costs) / len(costs)) if costs else None,
    }


## Grade the improved runtime and render the scorecard

Now we run the whole prod round end to end: `run_prod_round` invokes the deployed (improved,
redeployed) agent for every scenario, `grade_sessions` scores each one, and `build_scorecard`
folds the round into a single scorecard.

Read the scorecard as a parity check plus two prod-only signals. Correctness, ToolSelectionAccuracy
and OpsQuality confirm the Stage-1 prompt fix **did not break quality** — it holds in production,
just as it did locally. Latency and cost are what a local run can't produce: they are the price of
the real runtime, and the payoff of the trajectory fix measured where it actually runs. The
before→after narrative is therefore **Stage-1 local (naive prompt, flagged inefficient) → Stage-2
prod (improved prompt, graded on the live runtime)**.

> `run_prod_round` does a live AgentCore invocation per scenario and then waits ~180s for OTEL
> spans to land in CloudWatch, so this cell is slow. Run it once.


In [ ]:
from util.deployed_eval import render_scorecard

# Run the full prod round: invoke the deployed (improved) runtime, grade every session, and
# fold the round into one scorecard. run_prod_round does live invocations + a ~180s ingestion
# wait, so this is the slow cell — run it once.
improved_sessions = run_prod_round("improved", evaluation_scenarios)
improved_graded = grade_sessions(improved_sessions)
improved_scorecard = build_scorecard(improved_sessions, improved_graded)

METRIC_KEYS = ["Correctness", "ToolSelectionAccuracy", "OpsQuality", "latency_s", "cost_usd"]

print("=" * 42)
print(" STAGE 2 — IMPROVED RUNTIME (prod, redeployed)")
print("=" * 42)
print(render_scorecard(improved_scorecard, METRIC_KEYS))
print()

# Cross-stage before->after: Stage-1 LOCAL graded the naive prompt and its trajectory judge
# flagged the low-stock scenario as inefficient; Stage-2 PROD grades the improved prompt on
# the real runtime — the latency/cost above is what that fix buys, measured in production.
for scenario_name in local_results:
    local_traj = next(
        (r for r in local_results.get(scenario_name, [])
         if r.get("evaluatorId") == "StrandsEvals.TrajectoryEvaluator"),
        {},
    )
    traj_score = local_traj.get("value")
    if isinstance(traj_score, (int, float)) and traj_score < 0.5:
        # Both prod-only signals can be None if the whole round errored (build_scorecard
        # returns latency_s=None / cost_usd=None), so guard each before formatting.
        lat = improved_scorecard["latency_s"]
        cost = improved_scorecard["cost_usd"]
        lat_str = f"{lat:.1f}s" if isinstance(lat, (int, float)) else "n/a"
        cost_str = f"${cost:.4f}" if isinstance(cost, (int, float)) else "n/a"
        print(f"Cross-stage: Stage-1 local flagged '{scenario_name}' as inefficient "
              f"(TrajJudge {traj_score:.2f}). Stage-2 prod ran the fixed prompt above — "
              f"mean latency {lat_str}, cost {cost_str}.")

## Cleanup (optional)

In [ ]:
# Uncomment to clean up the custom evaluator created in this lab.
# The AgentCore Memory resource and the deployed runtime are owned by Labs 8 and 7 —
# clean those up from their respective notebooks.

# agentcore_control.delete_evaluator(evaluatorId=CUSTOM_EVALUATOR_ID)
# print(f"Deleted evaluator: {CUSTOM_EVALUATOR_ID}")


## Summary

You evaluated the warehouse agent in **two stages**, grounded in Anthropic's [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents).

**Grader taxonomy:** code-based (fast, objective), model-based / LLM-as-judge (flexible, nuanced),
human (gold standard, for calibration) — plus the outcome-vs-trajectory distinction.

**Stage 1 — Local evaluation with Strands Evals** (fast, cheap pre-deploy gate):
- Code-based `ToolCalled`; LLM-as-judge `TrajectoryEvaluator` (fed each call's OData arguments so it
  *names* the wasteful calls) and `ToolSelectionAccuracyEvaluator`, plus token counts as an
  informational efficiency signal — all routed through the same `SAPGenAIHubModel`.
- Caught a **correct-but-inefficient** answer (the low-stock query) that outcome graders score 1.0,
  then **applied the judge's fix and re-tested**: the improved prompt cut redundant calls and
  tokens while keeping the answer correct — the full measure → diagnose → fix → re-measure loop.

**Stage 2 — Deployed evaluation with AgentCore** (deploy-then-verify on the real runtime):
- **Shipped the Stage-1 fix**: redeployed the improved prompt in-place to the live AgentCore
  Runtime (memory-enabled, memory owned by Lab 9), because the fix had only ever lived locally.
- **Verified it in production** from OTEL traces — built-in evaluators (Correctness, Helpfulness,
  GoalSuccessRate, ToolSelectionAccuracy) plus a custom LLM-as-judge evaluator
  (WarehouseOperationalQuality) confirmed quality **held (parity)**, alongside **prod-only latency
  and cost** — the signals a local run physically can't produce.
- The before→after is therefore **cross-stage**: Stage-1 local (naive prompt, flagged inefficient)
  → Stage-2 prod (improved prompt, measured where it actually runs) — no separate prod baseline round.

**Key takeaway:** evaluate locally first, then ship the fix and verify it in production; combine
code-based and model-based graders so you measure not just *whether* the agent is right (outcome)
but *how efficiently* it gets there (trajectory) — and close the loop by fixing what the judge
surfaces and re-grading.

**Next steps (see the evaluation harness spec):** negative / out-of-scope scenarios, multi-trial
runs with pass@k / pass^k, chaos/fault-injection (Strands Evals 1.0+), and a pass/fail regression gate.
